# Bernstein–Vazirani Algorithm

This algorithm is used to figure out a hidden binary string using just one oracle query.

Compared to Deutsch–Jozsa, this algorithm feels more concrete because the final measurement directly reveals the hidden string encoded by the oracle.

# Main Idea

We have a hidden binary string.

Example: 1011

We don’t know this string beforehand.

The oracle uses this hidden string internally while computing:

f(x) = s · x mod 2

where:
- s is the hidden string
- x is the input string
- the dot product is calculated bit-by-bit
- mod 2 means the final answer becomes either 0 or 1

# Why Classical Computing Needs More Queries

Classically, we’d usually need multiple oracle queries to recover the hidden string.

For example:
- one query may reveal information about one bit
- another query reveals information about another bit

So for an n-bit string, we’d typically need n queries.

# What Quantum Computing Does Differently

Quantum computing creates a superposition of many input states at once.

The oracle then introduces phase information depending on the hidden string.

After the final Hadamard gates, interference happens in a way that reveals the hidden string directly.

# Oracle Construction

For every position where the hidden string contains 1, a CNOT gate is added.

Example: Hidden string = 1011

This means CNOT gates are added for:
- qubit 0
- qubit 2
- qubit 3

These gates encode information about the hidden string into phase.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# hidden binary string
secret_string = "1011"

n = len(secret_string)

# create circuit
qc = QuantumCircuit(n + 1, n)

# initialize helper qubit to |1>
qc.x(n)

# apply hadamard gates
for qubit in range(n + 1):
    qc.h(qubit)

# oracle construction
for i, bit in enumerate(secret_string):

    # if the bit is 1,
    # apply CNOT
    if bit == "1":
        qc.cx(i, n)

# apply hadamard again
# to input qubits
for qubit in range(n):
    qc.h(qubit)

# measure input qubits
for qubit in range(n):
    qc.measure(qubit, qubit)

# draw circuit
print(qc.draw())

# run simulator
simulator = AerSimulator()

job = simulator.run(qc, shots=1000)

result = job.result()
counts = result.get_counts()

print("hidden string:", secret_string)
print("measurement results:", counts)

# Observation

The final measurement becomes:

1011

which matches the hidden string used inside the oracle.

At first this felt a little strange because the algorithm never directly “reads” the hidden string bit-by-bit.

Instead:
- the oracle encodes information into phase
- the final Hadamard gates create interference
- the interference pattern reconstructs the hidden string during measurement

So rather than checking individual outputs one at a time, the algorithm uses phase kickback and interference to recover the full string in a single oracle query.